In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("ExpenseETL") \
    .getOrCreate()

expense_data = [
    (1, "2026-05-01", "Food", 250.0),
    (1, "2026-05-02", "Transport", 100.0),
    (1, "2026-05-03", "Shopping", 2000.0),
    (2, "2026-05-04", "Food", 300.0),
    (2, "2026-05-05", "Bills", 1500.0),
    (1, "2026-05-06", "Entertainment", 4500.0)
]

expense_columns = [
    "user_id",
    "date",
    "category",
    "amount"
]

expense_df = spark.createDataFrame(
    expense_data,
    expense_columns
)

print("Expense Data:")
expense_df.show()

user_data = [
    (1, "Viswanand", 25000),
    (2, "Arun", 18000)
]

user_columns = [
    "user_id",
    "user_name",
    "monthly_income"
]

user_df = spark.createDataFrame(
    user_data,
    user_columns
)

print("User Data:")
user_df.show()

expense_df = expense_df.withColumn(
    "expense_date",
    to_date(col("date"))
)

expense_df = expense_df.withColumn(
    "month",
    date_format(col("expense_date"), "yyyy-MM")
)

print("Expense Data With Month:")
expense_df.show()

combined_df = expense_df.join(
    user_df,
    on="user_id",
    how="inner"
)

print("Combined Data:")
combined_df.show()

summary_df = combined_df.groupBy(
    "user_id",
    "user_name",
    "month",
    "monthly_income"
).agg(
    sum("amount").alias("total_spend")
)


summary_df = summary_df.withColumn(
    "savings",
    col("monthly_income") - col("total_spend")
)

print("Summary With Savings:")
summary_df.show()

summary_df = summary_df.withColumn(
    "alert",
    when(
        col("total_spend") > col("monthly_income") * 0.7,
        "High Spending"
    ).otherwise("Normal")
)

print("Final Summary:")
summary_df.show()

summary_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("expense_summary_delta")

print("Delta table saved successfully.")

delta_df = spark.read.table("expense_summary_delta")

delta_df.show()

print("Reading Delta Table:")
delta_df.show()

Expense Data:
+-------+----------+-------------+------+
|user_id|      date|     category|amount|
+-------+----------+-------------+------+
|      1|2026-05-01|         Food| 250.0|
|      1|2026-05-02|    Transport| 100.0|
|      1|2026-05-03|     Shopping|2000.0|
|      2|2026-05-04|         Food| 300.0|
|      2|2026-05-05|        Bills|1500.0|
|      1|2026-05-06|Entertainment|4500.0|
+-------+----------+-------------+------+

User Data:
+-------+---------+--------------+
|user_id|user_name|monthly_income|
+-------+---------+--------------+
|      1|Viswanand|         25000|
|      2|     Arun|         18000|
+-------+---------+--------------+

Expense Data With Month:
+-------+----------+-------------+------+------------+-------+
|user_id|      date|     category|amount|expense_date|  month|
+-------+----------+-------------+------+------------+-------+
|      1|2026-05-01|         Food| 250.0|  2026-05-01|2026-05|
|      1|2026-05-02|    Transport| 100.0|  2026-05-02|2026-05|
|  